In [16]:
import json

import numpy as np
import pandas as pd
from tqdm import tqdm
from Bio import SeqIO

### vsearch

In [3]:
def get_sequence_lengths(fasta_file):
    seq_lengths = {}
    for record in SeqIO.parse(fasta_file, "fasta"):
        seq_lengths[record.id] = len(record.seq)  # 存储序列ID和长度
    return seq_lengths

fasta_path = "data/feces_seq_16S_SLIVA.fasta"
length_dict = get_sequence_lengths(fasta_path)

In [4]:
colnames = ["query_id", "refer_id", "identity", "alignment_length", "mismatches", "gap_openings", "q.start",
            "q.end", "s.start", "s.end", "e-value", "bit_score"]

In [7]:
vsearch_out = pd.read_csv("data/vsearch_blast.out", sep="\t", header=None)
vsearch_out.columns = colnames
vsearch_out.loc[:, "length"] = [length_dict[i] for i in vsearch_out.query_id.values]
vsearch_out.loc[:, "coverage"] = vsearch_out.alignment_length.values / vsearch_out.length.values

In [8]:
vsearch_out = vsearch_out.loc[vsearch_out["q.start"] == 1]
vsearch_out = vsearch_out.loc[vsearch_out["coverage"] >= 1]
vsearch_out.loc[:, "refer"] = [i.split("::")[1].split(":")[0] for i in vsearch_out.refer_id.values]

In [9]:
vsearch_out = vsearch_out.loc[vsearch_out.groupby('refer')['identity'].idxmax()]

In [10]:
vsearch_out.shape

(1125, 15)

In [11]:
def select_sequences_by_ids(input_fasta, output_fasta, selected_ids):

    selected_sequences = []
    for record in SeqIO.parse(input_fasta, "fasta"):
        if record.id in selected_ids:  
            record.id = record.id.split("::")[1].split(":")[0]
            selected_sequences.append(record)
    
    SeqIO.write(selected_sequences, output_fasta, "fasta")

target_id = vsearch_out.refer_id.values
input_fasta = "data/barrnap.fna"  
output_fasta = "data/pick_otu.fasta" 
select_sequences_by_ids(input_fasta, output_fasta, target_id) 

In [15]:
with open('data/fasta_dict.json', 'r', encoding='utf-8') as f:
    fasta_dict = json.load(f) 

In [17]:
target_genome = {}
for target in tqdm(vsearch_out.refer.values, desc="Processing"):
    for key, value in fasta_dict.items():
        if target in value:
            target_genome[target] = key
            continue

Processing: 100%|██████████| 1125/1125 [00:35<00:00, 31.36it/s]


In [19]:
vsearch_out.loc[:, "genome_id"] = [target_genome[i] for i in vsearch_out.refer.values]
vsearch_out = vsearch_out.loc[vsearch_out.groupby('genome_id')['identity'].idxmax()]

In [20]:
vsearch_out.shape

(1112, 16)

In [21]:
vsearch_out.head()

,query_id,refer_id,identity,alignment_length,mismatches,gap_openings,q.start,q.end,s.start,s.end,e-value,bit_score,length,coverage,refer,genome_id
60,AJ305238.1.1505,16S_rRNA::DS480346.1:176999-178547(+),99.9,1505,2,0,1,1505,1,1548,-1,0,1505,1.000000,DS480346.1,GCA_000154345.1_genomic
978,ABQR01000074.101.1610,16S_rRNA::DS990270.1:96-1623(+),100.0,1510,0,0,1,1510,1,1527,-1,0,1510,1.000000,DS990270.1,GCA_000155435.1_genomic
133,Y18181.1.1435,16S_rRNA::EQ973341.1:3511-5036(-),99.9,1437,0,2,1,1435,1,1525,-1,0,1435,1.001394,EQ973341.1,GCA_000158655.1_genomic
1900,ACON01000003.712136.713636,16S_rRNA::GG688422.1:1081954-1083464(+),100.0,1501,0,0,1,1501,1,1510,-1,0,1501,1.000000,GG688422.1,GCA_000161975.1_genomic
25,AJ239289.1.1355,16S_rRNA::ADBE01000137.1:300-1836(+),100.0,1355,0,0,1,1355,1,1536,-1,0,1355,1.000000,ADBE01000137.1,GCA_000176735.1_genomic


In [22]:
vsearch_out.to_csv("data/vsearch_res.csv", index=None)